#### FastAI UNet - Training
- Reference: https://www.youtube.com/watch?v=DKzL4zumFi8
- https://walkwithfastai.com/Segmentation

In [ ]:
import os
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES="4,5"

In [ ]:
import numpy as np
from pathlib import Path

from fastai.vision.all import *

path = Path("datasets/battus10/training_images")
path

In [ ]:
codes = np.loadtxt(os.path.join(path, "codes.txt"), dtype='str')
files = get_image_files(os.path.join(path, "images"))
print("Total Images:", len(files), " \t Sample: ", files[0])
print("Codes: ", codes)

In [ ]:
img = PILImage.create(files[1])
img.show(figsize=(5,5))

In [ ]:
def label_func(image):
    """Function used to label images while training. Required by fastai."""
    return os.path.join(path, "labels", f"{image.stem}.png")

msk = PILMask.create(label_func(files[1]))
msk.show(figsize=(5,5), alpha=1)

In [ ]:
name2id = {v:k for k,v in enumerate(codes)}
print("Unique labels:",np.unique(msk), "\n", name2id)

In [ ]:
dls = SegmentationDataLoaders.from_label_func(path, bs=2, fnames=files, 
                                            label_func=label_func, codes=codes, num_workers=4, 
                                            batch_tfms=[*aug_transforms(mult=1.0, do_flip=True, 
                                                                        flip_vert=True, max_rotate=10., 
                                                                        max_zoom=1.1,max_warp=0.2, 
                                                                        p_affine=0.75, p_lighting=0, 
                                                                        xtra_tfms=None)])
dls.show_batch()

## Train

In [ ]:
# Now, loading the model 
learn = unet_learner(dls, resnet34,  pretrained=True) # self_attention=True, weights="./init/segmentation_test-4classes.pkl"

In [ ]:
# Train for 20 epochs
learn.fine_tune(20)

In [ ]:
# Model export using pickle protocol - https://docs.fast.ai/learner.html#learner
learn.export('battus10_segmentation_test-4classes-resnet34-b2-e20.pkl')

In [ ]:
learn.show_results(max_n=2)